# 🟢 新手模式 | Beginner Mode

**不用写代码。** 拖动滑杆、点“运行实验”，看曲线和回放怎么变，再回答一个小问题，就完成一次探索。

**三步操作：**
1. 点菜单栏 Run ▶▶ Run All Cells（全部运行一次，等默认结果出来）；
2. 拖动下面的滑杆改一个关键参数；
3. 点 **▶ 运行实验**，看对比图和一句话解读。

**诚实说明：** 你看到的数字全部来自真实引擎（不是演示动画），和完整模式是同一口径。
新手模式只写 `labs/data/lite/` 下的完成记录，**不计入教师成绩册**；想拿实验成绩，请接着做对应的完整模式。

# C1 新手模式：车重一点、风阻大一点，电耗多多少？


## 直觉问题

同一辆纯电车（2017 Chevrolet Bolt）、同一条城市工况（UDDS，走走停停 17 分钟）：

- 后备箱多装 200kg，电耗多多少？
- 风阻系数从 0.29 变成 0.35，电耗又多多少？

本实验让你调两个数：**额外载重**和**风阻系数**。一次只动一个，先看单因素的影响。

说明：FASTSim 引擎住在独立的 Python 3.10 环境里（那个环境没有滑杆组件），
本本通过一个小脚本调用它，结果口径和完整模式一致（D3 电池链路关闭，只看车重/风阻）。

**你的任务在最后**：只加 200kg，和只把风阻调到 0.35，哪个更费电？


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
sys.dont_write_bytecode = True

# 新手驱动：滑杆 -> 真实引擎 -> 对比图（引擎本身不在本本里，全部复用完整模式的同一批代码）
from common.lite_drivers import (
    plot_curves, plot_bars,
)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

LAST = {}  # 最近一次运行的结果（供“保存心得”格子使用）
print("✓ 新手驱动就绪：下面拖动滑杆、点运行即可")


In [ ]:
from common.lite_drivers import plot_bars, interpret_c1, save_lite_record, render_lite_metric_card
import json, subprocess

VNUM, BASE_DRAG = 17, 0.29  # 2017 Chevrolet Bolt（纯电，风阻系数 0.29）
PY310 = "/opt/envs/py310-fastsim/bin/python"
RUNNER = "/home/jovyan/labs/common/c1_lite_runner.py"

def _run_fastsim(vnum, kg_v, cd_v):
    r = subprocess.run(
        [PY310, RUNNER, "--vnum", str(vnum), "--kg", str(kg_v), "--cd", str(cd_v)],
        capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        raise RuntimeError(f"FASTSim 调用失败：{r.stderr[-500:]}")
    return float(json.loads(r.stdout.strip())["kwh_per_100km"])

_BASE = {}

kg = widgets.IntSlider(
    value=0, min=0, max=400, step=20,
    description="额外载重 (kg)：", style={"description_width": "120px"},
    layout=widgets.Layout(width="70%"),
)
cd = widgets.FloatSlider(
    value=0.29, min=0.20, max=0.40, step=0.01,
    description="风阻系数 Cd：", style={"description_width": "120px"},
    layout=widgets.Layout(width="70%"),
)
run_btn = widgets.Button(description="▶ 运行实验", button_style="success")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output(wait=True)
        print(f"正在跑真 FASTSim 仿真（Bolt + {kg.value}kg，Cd={cd.value:.2f}）……请稍等")
        if "kwh" not in _BASE:
            _BASE["kwh"] = _run_fastsim(VNUM, 0, BASE_DRAG)
        base_kwh = _BASE["kwh"]
        stu_kwh = _run_fastsim(VNUM, kg.value, cd.value)
        out.clear_output(wait=True)
        
        display(plot_bars(
            ["基线工况 (0kg, Cd=0.29)", f"当前设置 (+{kg.value}kg, Cd={cd.value:.2f})"],
            [base_kwh, stu_kwh],
            "百公里综合电耗 (kWh/100km)", "UDDS 城市工况百公里电耗对比"))
        
        pct = (stu_kwh - base_kwh) / base_kwh * 100 if base_kwh else 0
        badge_text = f"⚡ 能耗增加: +{pct:.1f}%" if pct > 0 else (f"🌱 能耗降低: {abs(pct):.1f}%" if pct < 0 else "⚪ 基线工况")
        badge_color = "red" if pct > 5 else ("green" if pct <= 0 else "yellow")
        metrics_list = [
            ("百公里综合电耗", f"{stu_kwh:.2f} kWh"),
            ("基线能耗参考", f"{base_kwh:.2f} kWh"),
            ("额外载荷变化", f"+{kg.value} kg"),
            ("风阻系数", f"{cd.value:.2f}"),
        ]
        display(render_lite_metric_card("C1 纯电能耗", badge_text, badge_color, metrics_list, interpret_c1(base_kwh, stu_kwh)))
        
        save_lite_record("C1", {"extra_kg": kg.value, "drag_coef": cd.value},
                         {"kwh_per_100km": stu_kwh})
        LAST.clear()
        LAST.update({"exp": "C1",
                     "params": {"extra_kg": kg.value, "drag_coef": cd.value},
                     "metrics": {"kwh_per_100km": stu_kwh}})

run_btn.on_click(on_run)
display(kg, cd, run_btn, out)
on_run(None)

## 🎯 找区间小任务

先一次只动一个滑杆，回答下面两个问题（写在下一个格子的心得里）：

1. 只加 200kg 载重，电耗变化百分之几？只把风阻从 0.29 调到 0.35，变化百分之几？哪个更费电？
2. （进阶）两个同时调到最大，电耗变化是不是等于两个单因素变化之和？为什么？


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from common.lite_drivers import save_lite_record

note = widgets.Textarea(
    value="",
    placeholder="写一句话：你找到了什么规律？（例如：阈值越大，刹车越早，但……）",
    description="我的心得：",
    layout=widgets.Layout(width="100%", height="80px"),
)
save_btn = widgets.Button(description="💾 保存心得与打卡记录", button_style="info")
save_out = widgets.Output()

def _on_save(b):
    with save_out:
        save_out.clear_output(wait=True)
        if not LAST:
            print("⚠️ 请先运行一次实验（上一个格子的“运行实验”按钮），再保存。")
            return
        ans = note.value.strip()
        if not ans:
            print("⚠️ 请在上方输入框写下一句寻优心得后再打卡保存！")
            return
        path = save_lite_record(LAST["exp"], LAST["params"], LAST["metrics"], note=ans)
        display(HTML("<span style='color: #16a34a; font-weight: bold;'>🎉 恭喜！新手探索打卡成功，已同步归档入预习数据库！</span><br/><span style='color: #64748b; font-size: 12px;'>注：新手模式完成记录独立于 29+1 课程成绩册，不影响期末考核。</span>"))

save_btn.on_click(_on_save)
display(note, save_btn, save_out)

---

## 下一步 | Next

- 完成上面的“找区间”小任务后，把你的心得保存下来（上一个格子）。
- 想拿实验成绩、学完整算法：去做对应的完整模式（同目录下不带 `_lite` 的 notebook）。
- 新手模式的意义是先建立直觉：参数变了、结果往哪个方向变、变化有多大。

完整模式：`C1_fastsim_energy_pro.ipynb`
